##Import Necessary Libraries

In [ ]:
import warnings
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import math
import tensorflow as tf
import glob
import shutil
import zipfile
from google.colab import drive
import xml.etree.ElementTree as ET
from collections import Counter
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import cv2
import albumentations as A

warnings.filterwarnings('ignore')


In [ ]:
!pip install ultralytics --quiet

In [ ]:
!pip install -q albumentations opencv-python

##Connect google drive

In [ ]:
from google.colab import drive
#Mount Google Drive
drive.mount('/content/drive')

In [ ]:
# Unzip the Dataset
# !unzip "/content/drive/MyDrive/newest_dataset.zip" -d "/content/drive/MyDrive/road_data_4/"

In [ ]:
# Unzip the Labels
# !unzip "/content/drive/MyDrive/road_dataset_labels2.zip" -d "/content/drive/MyDrive/road_data_4/"

##Find Images with Missing Labels

In [ ]:
import os
import glob

img_dir = '/content/drive/MyDrive/road_data_4/newest_dataset'
lbl_dir = '/content/drive/MyDrive/road_data_4/road_dataset_labels2'

IMAGE_EXTENSIONS = ['*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG']

# 1. Collect all image paths
image_paths = []
for ext in IMAGE_EXTENSIONS:
    image_paths.extend(glob.glob(os.path.join(img_dir, '**', ext), recursive=True))

# 2. Collect all label filenames without extension (stems)
label_stems = {
    os.path.splitext(os.path.basename(f))[0]
    for f in glob.glob(os.path.join(lbl_dir, '**', '*.txt'), recursive=True)
}

# 3. Find images missing matching .txt label files
missing_images = [
    img for img in image_paths
    if os.path.splitext(os.path.basename(img))[0] not in label_stems
]

# 4. Print results
print(f"Found {len(missing_images)} images without labels:\n")
for img_path in missing_images:
    print(img_path)

##Generating Empty Labels For them

In [ ]:
img_dir = '/content/drive/MyDrive/road_data_4/newest_dataset'
lbl_dir = '/content/drive/MyDrive/road_data_4/road_dataset_labels2'

# Ensure the labels directory exists
os.makedirs(lbl_dir, exist_ok=True)

IMAGE_EXTENSIONS = ['*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG']

# Gather all image paths
image_paths = []
for ext in IMAGE_EXTENSIONS:
    image_paths.extend(glob.glob(os.path.join(img_dir, '**', ext), recursive=True))

# Save matching empty .txt files directly inside the label folder
created_count = 0
for img_path in image_paths:
    file_name = os.path.splitext(os.path.basename(img_path))[0]
    target_txt_path = os.path.join(lbl_dir, f"{file_name}.txt")

    if not os.path.exists(target_txt_path):
        open(target_txt_path, 'w').close()
        created_count += 1

# Audit final counts
total_imgs = len(image_paths)
total_lbls = len(glob.glob(os.path.join(lbl_dir, '**', '*.txt'), recursive=True))

print(f"Created {created_count} empty files in: {lbl_dir}")
print(f"Total Images: {total_imgs} | Total Labels: {total_lbls}")

##Confirming The Classes

In [ ]:
dataset_root_dir = '/content/drive/MyDrive/road_data_4/road_dataset_labels2'

# Search for classes.txt
classes_txt_path = None
for root, _, files in os.walk(dataset_root_dir):
    if 'classes.txt' in files:
        classes_txt_path = os.path.join(root, 'classes.txt')
        break


if classes_txt_path:
    print(f"Found classes.txt at: {classes_txt_path}\n")
    print("Content of classes.txt:")
    with open(classes_txt_path, 'r') as f:
        print(f.read())
else:
    print("Classes file not found")

##Data Visualization

In [ ]:
import matplotlib.pyplot as plt
import cv2

TARGET_SIZE = (160, 160)

# Select up to 4 images from our scanned list to verify visually
sample_images = image_paths[:4]

if len(sample_images) == 0:
    print("Error: No images available to visualize. Ensure your Drive folder is loaded.")
else:
    # Set up a clean horizontal grid layout in Matplotlib
    fig, axes = plt.subplots(1, len(sample_images), figsize=(15, 5))

    # Handle edge case where only 1 image exists (axes wouldn't be an array)
    if len(sample_images) == 1:
        axes = [axes]

    print(f"Processing and rendering uniform {TARGET_SIZE[0]}x{TARGET_SIZE[1]} pixel maps...")

    for idx, img_path in enumerate(sample_images):
        # Load the raw image file via OpenCV
        raw_img = cv2.imread(img_path)

        # Convert from BGR to RGB
        rgb_img = cv2.cvtColor(raw_img, cv2.COLOR_BGR2RGB)

        # Apply uniform spatial pixel reduction
        resized_img = cv2.resize(rgb_img, TARGET_SIZE, interpolation=cv2.INTER_AREA)

        # Render to the grid layout
        axes[idx].imshow(resized_img)
        axes[idx].set_title(f"Sample {idx+1}\nShape: {resized_img.shape}", fontsize=10)
        axes[idx].axis('off')  # Hide pixel grid coordinates for a clean look

    plt.tight_layout()
    plt.show()

##Class Distribution Analysis

In [ ]:
# Define the absolute path to dataset folder inside Google Drive
DRIVE_DATASET_PATH = 'content/drive/MyDrive/road_data_4/newest_dataset'

# Define the path to classes file inside your labels directory
# Adjust 'labels/classes.txt' if your file is placed or named slightly differently
CLASSES_TXT_PATH = os.path.join('/content/drive/MyDrive/road_data_4/road_dataset_labels2', 'classes.txt')


YOLO_CLASS_MAPPING = {}

if os.path.exists(CLASSES_TXT_PATH):
    with open(CLASSES_TXT_PATH, 'r') as f:
        lines = f.readlines()
        for idx, line in enumerate(lines):
            clean_name = line.strip().lower()
            if clean_name: # Ignore empty lines
                YOLO_CLASS_MAPPING[idx] = clean_name

    print(" Successfully found and imported your labelImg classes file!")
    print("Automatically generated class map index:")
    for index, name in YOLO_CLASS_MAPPING.items():
        print(f"   ID {index} ➔ Class Name: '{name}'")
else:
    print("Could not automatically find 'classes.txt'.")


##Count The Dataset

In [ ]:
import os
import glob
import matplotlib.pyplot as plt
from collections import Counter

# 1. Point directly to your newly combined unified master labels folder
FINAL_LABEL_DIR = "/content/drive/MyDrive/road_data_4/road_dataset_labels2"

class_counter = Counter()
unlabeled_count = 0

# Find all consolidated text files
label_files = glob.glob(os.path.join(FINAL_LABEL_DIR, "*.txt"))

# Fixed print statement: Uses len(label_files) instead of len(FINAL_LABEL_DIR)
print(f"Scanning {len(label_files)} combined annotation files to calculate final distributions...")

# Define the absolute class index dictionary matching your final integer outputs
YOLO_CLASS_MAPPING = {
    0: "cracks",
    1: "speed_bumps",
    2: "potholes",
    3: "left_road_boundaries",
    4: "right_road_boundaries"
    ,
}

# PARSING LOOP
for anno_path in label_files:
    if not os.path.exists(anno_path) or os.path.getsize(anno_path) == 0:
        unlabeled_count += 1
        continue

    try:
        with open(anno_path, "r") as f:
            lines = f.readlines()

        if not lines or len(lines) == 0:
            unlabeled_count += 1
            continue

        for line in lines:
            parts = line.strip().split()
            if len(parts) > 0:
                raw_class = parts[0]
                try:
                    class_id = int(raw_class)
                    class_name = YOLO_CLASS_MAPPING.get(class_id, f"unknown_id_{class_id}")
                except ValueError:
                    class_name = raw_class

                class_counter[class_name] += 1
    except Exception:
        pass

# ADDITION: Include empty background images into the visual counter
if unlabeled_count > 0:
    class_counter["background_images"] = unlabeled_count

# Sort the items dynamically from highest instance count to lowest
sorted_classes = sorted(class_counter.items(), key=lambda x: x[1], reverse=True)

if len(sorted_classes) == 0:
    print("\n No objects found inside your final annotation files. Check your folder directory path.")
else:
    labels, counts = zip(*sorted_classes)

    print("\n FINAL UNIFIED DATASET DISTRIBUTION LOG (SORTED)")
    print("=" * 50)
    for class_name, count in sorted_classes:
        print(f"  ↳ {class_name.ljust(25)} : {count} instances")
    print("-" * 50)
    print(f" Total Unique Images Configured     : {len(label_files)}")
    print(f" Empty/Unlabeled Text Files         : {unlabeled_count}")
    print("=" * 50)

    # VISUALIZATION LAYOUT
    plt.figure(figsize=(10, 6))

    # Optional: Highlight background images with a distinct color (#e74c3c)
    bar_colors = ["#e74c3c" if label == "background_images" else "#2ecc71" for label in labels]

    bars = plt.bar(labels, counts, color=bar_colors, edgecolor="#27ae60", width=0.55)

    # Superimpose numeric count values cleanly on top of each bar
    for bar in bars:
        height = bar.get_height()
        plt.annotate(
            f"{height}",
            xy=(bar.get_x() + bar.get_width() / 2, height),
            xytext=(0, 4),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=10,
            fontweight="bold",
        )

    plt.title(
        "Final Unified Dataset Class Distribution Analysis",
        fontsize=14,
        fontweight="bold",
        pad=15,
    )
    plt.xlabel("Class Categories", fontsize=12, labelpad=10)
    plt.ylabel("Total Combined Label Instance Count", fontsize=12, labelpad=10)
    plt.xticks(rotation=15, fontsize=10)
    plt.yticks(fontsize=11)
    plt.grid(axis="y", linestyle="--", alpha=0.4)

    plt.tight_layout()
    plt.show()

##Split The Data

In [ ]:
import os
import glob
import random
import shutil
from tqdm import tqdm


# 1. Paths to your raw source folders
ORIG_IMG_DIR = "/content/drive/MyDrive/road_data_4/newest_dataset"
ORIG_LBL_DIR = "/content/drive/MyDrive/road_data_4/road_dataset_labels2"

# 2. Output directory under your master road_data folder
BASE_OUTPUT_DIR = "/content/drive/MyDrive/road_data_4/split_dataset2/"

splits = ['train', 'val', 'test']
for split in splits:
    os.makedirs(os.path.join(BASE_OUTPUT_DIR, split, 'images'), exist_ok=True)
    os.makedirs(os.path.join(BASE_OUTPUT_DIR, split, 'labels'), exist_ok=True)

# 3. Match and collect your raw source file pairs
all_pairs = []

orig_labels = glob.glob(os.path.join(ORIG_LBL_DIR, "*.txt"))
for lbl_path in orig_labels:
    base = os.path.basename(lbl_path).replace(".txt", "")

    # Check for .jpg or .png formats dynamically
    img_path = os.path.join(ORIG_IMG_DIR, f"{base}.jpg")
    if not os.path.exists(img_path):
        img_path = os.path.join(ORIG_IMG_DIR, f"{base}.png")

    if os.path.exists(img_path):
        all_pairs.append({
            'img': img_path,
            'lbl': lbl_path
        })

print(f" Found {len(all_pairs)} matching source image/label pairs to process.")

#  Shuffle mathematically for random distribution
random.seed(42)
random.shuffle(all_pairs)

#  Calculate split point boundaries based on 70% / 20% / 10%
total_count = len(all_pairs)
train_end = int(total_count * 0.70)
val_end = train_end + int(total_count * 0.20)

# 6. Distribute files to their new home locations
copied_tracker = {'train': 0, 'val': 0, 'test': 0}

for idx, pair in enumerate(tqdm(all_pairs, desc="Splitting Dataset")):
    if idx < train_end:
        chosen_split = 'train'
    elif idx < val_end:
        chosen_split = 'val'
    else:
        chosen_split = 'test'

    # Get exact file extensions
    img_ext = os.path.splitext(pair['img'])[1]
    base_filename = os.path.basename(pair['lbl']) # Keep original filename cleanly
    img_filename = base_filename.replace(".txt", img_ext)

    dest_img_path = os.path.join(BASE_OUTPUT_DIR, chosen_split, 'images', img_filename)
    dest_lbl_path = os.path.join(BASE_OUTPUT_DIR, chosen_split, 'labels', base_filename)

    # Direct high-speed file transfer copy
    shutil.copy(pair['img'], dest_img_path)
    shutil.copy(pair['lbl'], dest_lbl_path)

    copied_tracker[chosen_split] += 1


print(" SOURCE DATASET SPLIT COMPLETE!")

print(f"  Train folder (70%) : {copied_tracker['train']} pairs")
print(f"  Val folder   (20%) : {copied_tracker['val']} pairs")
print(f"  Test folder  (10%) : {copied_tracker['test']} pairs")

print(f" Your data structure is completely ready inside: {BASE_OUTPUT_DIR}")

##Create A Yaml file

In [ ]:
import yaml
import os
from ultralytics import YOLO

yolo_yaml = {
    'path': '/content/drive/MyDrive/road_data_4/split_dataset2', # Changed: Path to the root of the split dataset
    'train': 'train/images',
    'val': 'val/images',
    'test': 'test/images',
    'nc': 5,  # Corrected to match the number of names
    'names': {
        0: 'cracks',
        1: 'speed_bumps',
        2: 'potholes',
        3: 'left_road_boundaries',
        4: 'right_road_boundaries',
           }
}

# Save it to drive
yaml_path = '/content/drive/MyDrive/road_data_4/autonomous_vehicle.yaml'

# Ensure the directory exists before writing the file
os.makedirs(os.path.dirname(yaml_path), exist_ok=True)

with open(yaml_path, 'w') as f:
    yaml.dump(yolo_yaml, f, default_flow_style=False)

print(f"Created YOLO configuration file at: {yaml_path}")

##Load pretrained weights for yolo11n

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11n.pt')

##Train The Model

In [ ]:
results = model.train(
    data='/content/drive/MyDrive/road_data_4/autonomous_vehicle.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,
    project='/content/drive/MyDrive/road_data_4/yolo_output',
    name='anomaly_detection_yolo11n',
    exist_ok=True
)

print("Model 1 training completed! Checkpoint saved in Google Drive.")

##Visualize The Trained Model

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/road_data_4/yolo_output/anomaly_detection_yolo11n/results.csv")
df.head()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Advanced Training Analytics

def plot_training_analytics(results_csv_path):
    """Loads results.csv and generates comprehensive performance visualizations."""

    try:
        df = pd.read_csv(results_csv_path)
        # Clean up whitespace in column names
        df.columns = df.columns.str.strip()
    except Exception as e:
        print(f"Error loading CSV: {e}")
        return

    # Training & Validation Losses
    fig1, axes1 = plt.subplots(2, 3, figsize = (15, 8))
    fig1.suptitle('Training and Validation Losses per Epoch', fontsize = 16)

    # Mapping columns to axes
    metrics = [('train/box_loss', 'Train Box Loss', 'tab:blue'),
               ('train/cls_loss', 'Train Cls Loss', 'tab:green'),
               ('train/dfl_loss', 'Train DFL Loss', 'tab:red'),
               ('val/box_loss', 'Val Box Loss', 'tab:blue'),
               ('val/cls_loss', 'Val Cls Loss', 'tab:green'),
               ('val/dfl_loss', 'Val DFL Loss', 'tab:red')]

    for i, (col, title, color) in enumerate(metrics):
        ax = axes1[i // 3, i % 3]
        ax.plot(df['epoch'], df[col], 'o-' if i < 3 else 's--', label = col, color = color)
        ax.set_title(title); ax.legend(); ax.grid(True)
        ax.set_xlabel('Epoch')

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

    # Performance Metrics
    fig2, axes2 = plt.subplots(2, 2, figsize=(15, 8))
    fig2.suptitle('Performance Metrics per Epoch', fontsize=16)

    metric_cols = [('metrics/precision(B)', 'Precision', 'tab:purple'),
                   ('metrics/recall(B)', 'Recall', 'tab:orange'),
                   ('metrics/mAP50(B)', 'mAP@50', 'tab:green'),
                   ('metrics/mAP50-95(B)', 'mAP@50:95', 'tab:red')]

    for i, (col, title, color) in enumerate(metric_cols):
        ax = axes2[i // 2, i % 2]
        ax.plot(df['epoch'], df[col], 'o-', label=title, color=color)
        ax.set_title(title); ax.legend(); ax.grid(True)
        ax.set_xlabel('Epoch')

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

plot_training_analytics("/content/drive/MyDrive/road_data_4/yolo_output/anomaly_detection_yolo11n/results.csv")

In [ ]:
import glob
from IPython.display import Image, display
from pathlib import Path

def display_training_results():
    """Displays all performance plots generated by the YOLO training process."""

    results_path = Path("/content/drive/MyDrive/road_data_4/yolo_output/anomaly_detection_yolo11n")

    print(f"\nDisplaying training results from: {results_path}")

    result_images = glob.glob(str(results_path/"*.png")) + glob.glob(str(results_path/"*.jpg"))

    for img_path in result_images:
        print(f"\n--- {Path(img_path).name} ---")
        display(Image(filename=img_path))

display_training_results()

##Train with YOLO26Nano

In [ ]:
model = YOLO('yolo26n.pt')

In [ ]:

# Train on road anomalies dataset for 60 epochs
results = model.train(
    data='/content/drive/MyDrive/road_data_4/autonomous_vehicle.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,
    project='/content/drive/MyDrive/road_data_4/yolo_output',
    name='anomaly_detection_yolo26n',
    exist_ok=True
)

print("Model 1 training completed! Checkpoint saved in Google Drive.")